# Imports

In [1]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

# Load Data

In [2]:
vcf = VCF("../data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("../data/ps3_gwas.phen", sep="\t", header=None)

In [4]:
pcs = pd.read_csv("../plink_results/ps3_gwas.eigenvec", sep=r"\s+", header=None)

# Clean and Align Phenotype Data

In [5]:
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# GWAS w/out PCA

In [6]:
# Set up output file
with open("../python_results/gwas_results.tsv", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    for v in vcf:
        # Convert genotype to dosage-like numeric
        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y)
        g2 = g[mask]
        y2 = y[mask]

        # Flip allele to match PLINK using minor allele, not ALT allele as effect allele
        if g2.mean() / 2 > 0.5:
            g2 = 2.0 - g2
            
        # Skip if not enough samples
        if len(g2) < 3:
            continue

        # MAF on non-missing genotypes
        maf = g2.mean() / 2
        if maf < 0.05:
            continue

        # Regression on filtered samples
        y0 = y2 - y2.mean()
        g0 = g2 - g2.mean()
        df = len(y2) - 2

        denom = g0 @ g0
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g0 @ y0) / denom

        resid = y0 - beta * g0
        se = np.sqrt((resid @ resid) / df / denom)

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        # Check SNPs
        snp_id = v.ID if v.ID is not None else f"{v.CHROM}:{v.POS}"

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{snp_id}\t{beta}\t{p}\t{len(y2)}\n")

# Clean and Align PC Data

In [7]:
pcs.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, pcs.shape[1] - 1)]
pcs = pcs.set_index("IID")
pc_matrix = pcs.loc[samples].drop(columns=["FID"]).values.astype(float)

intercept = np.ones(len(pc_matrix))
C = np.column_stack([intercept, pc_matrix])

# GWAS w/PCA

In [8]:
# Reload VCF
vcf = VCF("../data/ps3_gwas.vcf.gz")

In [9]:
# Compute y-residuals
covar_coefs = np.linalg.solve(C.T @ C, C.T @ y)
y_resid = y - C @ covar_coefs

# Precompute to save computation later
covar_complete = ~np.any(np.isnan(C), axis=1)

In [10]:
# Set up output file
with open("../python_results/gwas_results_covar.tsv", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    for v in vcf:
        # Convert genotype to dosage-like numeric
        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y) & covar_complete
        g2 = g[mask]
        y2 = y_resid[mask]
        C2 = C[mask]

        # Flip allele to match PLINK using minor allele, not ALT allele as effect allele
        if g2.mean() / 2 > 0.5:
            g2 = 2.0 - g2

        # Skip if not enough samples
        if len(g2) < C2.shape[1] + 2:
            continue

        # Compute g-residuals
        covar_coefs_g = np.linalg.solve(C2.T @ C2, C2.T @ g2)
        g_resid = g2 - C2 @ covar_coefs_g 

        # MAF on non-missing genotypes
        maf = g2.mean() / 2
        if maf < 0.05:
            continue

        # Regression on filtered samples
        denom = g_resid @ g_resid
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g_resid @ y2) / denom

        df = len(y2) - C2.shape[1] - 1
        resid = y2 - beta * g_resid
        se = np.sqrt((resid @ resid) / df / denom)

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        # Check SNPs
        snp_id = v.ID if v.ID is not None else f"{v.CHROM}:{v.POS}"

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{snp_id}\t{beta}\t{p}\t{len(y2)}\n")